In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import beamforming.read_sims_module as rm
from beamforming.adf_weights import compute_adf_weights, ADF_parameters, shower_direction_vector
from beamforming.utils import Bn
import beamforming.beamforming_module_para as bm
import beamforming.sampling_module_para as sp

R2D = 180/np.pi

SMALL_SIZE = 10
MEDIUM_SIZE = 12
BIGGER_SIZE = 14

plt.rc('font', size=BIGGER_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=MEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=MEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)

In [ ]:
class Config:
    """
    Configuration class to hold all parameters for the reconstruction process.
    """
    path_to_library: str = "/volatile/home/af274537/Documents/WorkingDir/HERON/SphericalPhasing_2/data/TauLibrary_972Events_Eshower_2e7-1e9GeV_XYZCoordinates_CorrXmax.npz"
    save_path: str = './results_adf'
    sampling: str = 'random'
    n_walkers: int = 200
    n_steps: int = 400
    step_size: float = 1000.
    f_min: float = 30
    f_max: float = 250
    noise_std: float = 13
    jitter_std: float = 0
    bounds: str = 'flat_sphere'  # Options: 'cubic', 'sphere', 'flat_sphere'
    r: float = 10.e3  # Radius for spherical bounds
    thickness: float = 4e3  # Thickness for flat spherical bounds
    intens_method: str = 'amplitude'  # Intensity calculation method
    temp: float = 5000.  # Temperature for sampling
    x_max_method: str = 'max'  # Method for finding maximum
    n_best_walkers: int = 0  # Number of best walkers to consider
    sep_walkers: bool = False  # Separate walkers flag
    burn_in: int = 0  # Number of burn-in steps to discard
config = Config()
name_prefix = "LEevents_"
tau_events, antennas, efields, sttimes = rm.read_library(config.path_to_library)

In [ ]:
efields.shape[0]

## 1.Finding outliers

In [ ]:
rec = pd.read_csv('results_adf_0402/LEevents_reconstruction_results.csv')

In [ ]:
plt.hist(rec['angular_error_deg'], bins=40, range=(0,5));


In [ ]:
plt.hist(rec['angular_error_deg'], bins=40, range=(0,5));

In [ ]:
plt.scatter(rec['max_intensity'], rec['angular_error_deg'])

In [ ]:
best_id = rec.sort_values('angular_error_deg').event_id.values[0]
map_best = np.load(f'results_adf_0402/LEevents_{best_id:.0f}_intensity_map.npy')
plt.imshow(map_best, vmin=0, vmax=0.1);
plt.colorbar();

In [ ]:
range_phi = np.linspace(-4, 4, 100)
range_theta = np.linspace(-4, 4, 100)

In [ ]:
worst_id = rec.sort_values('angular_error_deg').event_id.values[25]
# worst_id = 25
map_worst = np.load(f'results_adf_0402/LEevents_{worst_id:.0f}_intensity_map.npy')
plt.pcolormesh(range_phi, range_theta, map_worst, shading='auto')
plt.title(f'Intensity Map for Event {worst_id} (Angular Error: {rec.loc[rec.event_id == worst_id, "angular_error_deg"].values[0]:.2f} °)')
plt.colorbar();

In [ ]:
Eventlist = [worst_id]  # List of events to process
Nevents = len(Eventlist)
(azim, zen, en_nu, en_tau, tau_pos, xmax_pos, event_efield, antenna_pos, 
         xant, yant, zant, signals, phi, theta, shower_dir) = rm.read_trace(
                Eventlist[0], tau_events, antennas, efields, sttimes, config.f_min, config.f_max, config.noise_std, config.jitter_std
            )

In [ ]:
amps = np.max( np.linalg.norm(signals[1:], axis=0), axis=-1)
# plt.scatter(xant/1e3, yant/1e3, c=amps, cmap='viridis', s=20)
end_point = tau_pos + shower_dir*np.linalg.norm(tau_pos)
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
fig.suptitle(f'Antenna Layout with Signal Amplitudes for Event {worst_id}, $\\theta$={theta*R2D:.1f}°, $\\phi$={phi*R2D:.1f}°')

# sca = ax[0].scatter(xant/1e3, yant/1e3, c=10*np.log10(amps), cmap='viridis', s=20)
sca = ax[0].scatter(xant/1e3, yant/1e3, c=amps, cmap='viridis', s=20)
ax[0].scatter(xmax_pos[0]/1e3, xmax_pos[1]/1e3, c='red', marker='*', s=200, label='Xmax Position')
ax[0].scatter(tau_pos[0]/1e3, tau_pos[1]/1e3, c='red', marker='*', s=200, label='Xmax Position')
ax[0].plot([tau_pos[0]/1e3, end_point[0]/1e3], [tau_pos[1]/1e3, end_point[1]/1e3], c='blue', linestyle='--', label='Shower Direction')
ax[0].set_xlabel('X Antenna Position (km)')
ax[0].set_ylabel('Y Antenna Position (km)')
fig.colorbar(label='Max Signal Amplitude', ax=ax[0], mappable=sca)

ax[1].scatter(xant/1e3, zant/1e3, c=amps, cmap='viridis', s=20)
ax[1].scatter(xmax_pos[0]/1e3, xmax_pos[2]/1e3, c='red', marker='*', s=200, label='Xmax Position')
ax[1].scatter(tau_pos[0]/1e3, tau_pos[2]/1e3, c='red', marker='*', s=200, label='Xmax Position')
ax[1].plot([tau_pos[0]/1e3, end_point[0]/1e3], [tau_pos[2]/1e3, end_point[2]/1e3], c='blue', linestyle='--', label='Shower Direction')
ax[1].set_xlabel('X Antenna Position (km)')
ax[1].set_ylabel('Z Antenna Position (km)')
fig.colorbar(label='Max Signal Amplitude', ax=ax[1], mappable=sca)
plt.show()

plt.scatter(yant/1e3, zant/1e3, c=amps, cmap='viridis', s=20)
plt.scatter(xmax_pos[1]/1e3, xmax_pos[2]/1e3, c='red', marker='*', s=200, label='Xmax Position')
plt.scatter(tau_pos[1]/1e3, tau_pos[2]/1e3, c='red', marker='*', s=200, label='Xmax Position')
plt.plot([tau_pos[1]/1e3, end_point[1]/1e3], [tau_pos[2]/1e3, end_point[2]/1e3], c='blue', linestyle='--', label='Shower Direction')
plt.xlabel('Y Antenna Position (km)')
plt.ylabel('Z Antenna Position (km)')
plt.gca().set_aspect('equal')
plt.show()

In [ ]:
def smooth_adf(theta, phi, omega, omega_cr, Bn, eta, l_ant, delta_omega=0.1):

    K = shower_direction_vector(theta, phi)
   
    asym_coeff = -0.003*np.rad2deg(theta)+0.220

    asym = asym_coeff/np.sqrt(1. - np.dot(K,Bn)**2)

    adf = 1/l_ant / (1.+4.*( ((np.tan(omega)/np.tan(omega_cr))**2 - 1. )/delta_omega)**2)
    adf *= 1. + asym*np.cos(eta) # 
    return adf

In [ ]:
13*np.sqrt(3)

In [ ]:
eta, omega, omega_cr, l_ant, adf = ADF_parameters(theta, phi, delta_omega=1,
                                                  Xants=np.stack((xant, yant, zant), axis=-1),
                                                  Xsource=xmax_pos)
omega_smooth = np.linspace(-4, 4, 500)/R2D
adf_smooth = smooth_adf(theta, phi, omega_smooth, omega_cr[0], Bn, 0, 1, delta_omega=1.)
omega_signed = omega * np.sign(eta)
mask = amps > (13*np.sqrt(3))*.0
plt.scatter(omega_signed[mask]*R2D, amps[mask], c='blue', s=20)
sorted_omega_signed = np.argsort(omega_signed[mask])
factor = amps[mask].max()/adf_smooth.max()*1
plt.plot(omega_smooth*R2D, adf_smooth*factor+2, c='blue', linestyle='--', label='Sorted Amplitudes')
plt.axvline(omega_cr[0]*R2D, color='red', linestyle='--', label='Critical Angle')
plt.axvline(-omega_cr[0]*R2D, color='red', linestyle='--', label='Critical Angle')



## Fit worst with $w_{cr}$ = 1.5 

In [ ]:
signals[1, 2, :].std()

In [ ]:
plt.plot(signals[1, 0, :])


In [ ]:
for ant in range(signals.shape[1]):
    plt.plot(signals[1, ant, :])

In [ ]:
_tstep = signals[0,0,1] - signals[0,0,0]
_tarray = np.arange(0, signals.shape[-1]*_tstep, _tstep)
_tarray.shape

In [ ]:
shifted_times = bm.spherical_phasing((xmax_pos + shower_dir*10)[0], (xmax_pos + shower_dir*10)[1], (xmax_pos + shower_dir*10)[2], xant, yant, zant, signals[0, :, 0])
output = np.array(bm.signals_summing_adf(signals, shifted_times, weights=adf[None, :]))
for ant in range(signals.shape[1]):
    mask = (_tarray[0:] + shifted_times[0, ant] > 0) & (_tarray[0:] + shifted_times[0, ant] < np.inf)
    plt.plot((_tarray[0:] + shifted_times[0, ant])[mask], signals[1, ant, 0:][mask])

In [ ]:
np.sqrt(65)*13

In [ ]:
shifted_times = bm.spherical_phasing(xmax_pos[0], xmax_pos[1], xmax_pos[2], xant, yant, zant, signals[0, :, 0])
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
output = np.array(bm.signals_summing_adf(signals, shifted_times, weights=adf[None, :]))
polar = 1
peak_pos = np.abs(output[0,polar]).argmax()
mask_no_signal = (np.arange(output.shape[-1]) > peak_pos - 900) & (np.arange(output.shape[-1]) < peak_pos - 200)
ax[0].axvline(output[0,0,mask_no_signal][0], color='red', linestyle='--', label='No Signal Region')
ax[0].axvline(output[0,0,mask_no_signal][-1], color='red', linestyle='--', label='No Signal Region')
std = output[0,1, mask_no_signal].std()
amp = output[0,polar].max()
ax[0].plot(output[0,0], output[0,polar])
ax[0].set_title(f"ADF BeamForming : SNR={amp/std:.1f}")
ax[0].set_xlabel('Time (ns)')
ax[0].set_ylabel('Summed Signal Amplitude [a.u.]')
output = np.array(bm.signals_summing(signals, shifted_times))
std = output[0,polar, mask_no_signal].std()
amp = output[0,polar].max()
ax[1].plot(output[0,0], output[0,polar])
ax[1].set_title(f"Standard BeamForming: SNR={amp/std:.1f}")
ax[1].set_xlabel('Time (ns)')
ax[1].set_ylabel('Summed Signal Amplitude [a.u.]')
plt.tight_layout()

In [ ]:
for i in range(30):
    event_id = rec.sort_values('angular_error_deg').event_id.values[i]
    (azim, zen, en_nu, en_tau, tau_pos, xmax_pos, event_efield, antenna_pos, 
         xant, yant, zant, signals, phi, theta, shower_dir) = rm.read_trace(
                event_id, tau_events, antennas, efields, sttimes, config.f_min, config.f_max, config.noise_std, config.jitter_std
            )
    eta, omega, omega_cr, l_ant, adf = ADF_parameters(theta, phi, delta_omega=1,
                                                  Xants=np.stack((xant, yant, zant), axis=-1),
                                                  Xsource=xmax_pos)
    shifted_times = bm.spherical_phasing(xmax_pos[0], xmax_pos[1], xmax_pos[2], xant, yant, zant, signals[0, :, 0])
    fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
    output_adf = np.array(bm.signals_summing_adf(signals, shifted_times, weights=adf[None, :]))
    polar = 1
    peak_pos = np.abs(output_adf[0,polar]).argmax()
    mask_no_signal = (np.arange(output_adf.shape[-1]) > peak_pos - 900) & (np.arange(output_adf.shape[-1]) < peak_pos - 200)

    std_adf = output_adf[0,1, mask_no_signal].std()
    amp_adf = output_adf[0,polar].max()

    output = np.array(bm.signals_summing(signals, shifted_times))
    std = output[0,polar, mask_no_signal].std()
    amp = output[0,polar].max()

    ax[0].axvline(output_adf[0,0,mask_no_signal][0], color='red', linestyle='--', label='No Signal Region')
    ax[0].axvline(output_adf[0,0,mask_no_signal][-1], color='red', linestyle='--', label='No Signal Region')
    ax[0].plot(output_adf[0,0], output_adf[0,polar] * std/std_adf)
    ax[0].set_title(f"ADF BeamForming : SNR={amp_adf/std_adf:.1f}")
    ax[0].set_xlabel('Time (ns)')
    ax[0].set_ylabel('Summed Signal Amplitude [a.u.]')
    ax[1].plot(output[0,0], output[0,polar])
    ax[1].set_title(f"Standard BeamForming: SNR={amp/std:.1f}")
    ax[1].set_xlabel('Time (ns)')
    ax[1].set_ylabel('Summed Signal Amplitude [a.u.]')
    fig.suptitle(f'Event ID: {event_id}')
    plt.tight_layout()

In [ ]:
shifted_times = bm.spherical_phasing(xmax_pos[0], xmax_pos[1], xmax_pos[2], xant, yant, zant, signals[0, :, 0])
All_intensity = np.zeros((len(range_theta), len(range_phi)))
for i, theta_i in enumerate(range_theta):
    for j, phi_j in enumerate(range_phi):
        weights = compute_adf_weights(xmax_pos[0], xmax_pos[1], xmax_pos[2], theta+theta_i/R2D, phi+phi_j/R2D, xant, yant, zant, delta_omega=1)
        output = bm.signals_summing_ref_adf(signals, shifted_times, weights[None, :])
        norm_2 = output[0, 1, :]**2 + output[0, 2, :]**2 + output[0, 3, :]**2
        compute_intensity = sp.compute_amp if config.intens_method == 'amplitude' else sp.compute_power
        intensity = compute_intensity(norm_2)
        All_intensity[i, j] = intensity

In [ ]:
signals[1,:,:].shape

In [ ]:
plt.pcolormesh(range_phi, range_theta, All_intensity, shading='auto')
plt.title(f'Intensity Map for Event {worst_id} (Angular Error: {rec.loc[rec.event_id == worst_id, "angular_error_deg"].values[0]:.2f} °)')
plt.colorbar();